# 9. How deep should the tree grow?

**Open:** restart R; save the supplied script as `scripts/my-09-choose-tree.R`.

**Input:** Chapter 8's `tree-input.rds`. **Today:** repeat the fit/predict operations you know, using temporarily hidden training farms to choose depth. Keep the final test farms aside.

## 9.1 Make a guess before choosing

Chapter 8 fitted a tree with one question. A deeper tree can ask another question inside each group, creating smaller groups and more group averages.

**Write your guess:** would more questions improve the fit to known training records? Would they necessarily improve predictions for a new farm? Remember the invented 6-hour sample that broke our perfect hand-built rule.

Run `validation-package`, then the supplied `load-tree-input` block:

In [1]:
library(rpart)

In [2]:
source("scripts/tree_helpers.R")
if (!file.exists("outputs/checkpoints/tree-input.rds")) stop("Run Chapter 8 first to create tree-input.rds.")
tree_input <- readRDS("outputs/checkpoints/tree-input.rds")
train_survey <- tree_input$train

We will compare maximum depths **1, 2 and 3**. Run `depth-candidates`; `1:3` lists those three integers:

In [3]:
candidate_depths <- 1:3
candidate_depths

[1] 1 2 3

Depth counts questions along one root-to-leaf path. It does not count all the questions in the whole tree.

| Allowed depth | What it permits |
|---|---|
| 1 | One question, at most two final groups |
| 2 | A second question inside those groups, at most four final groups |
| 3 | A third question along a path, at most eight final groups |

The software can stop earlier; a depth limit is permission, not a requirement. It learns the cut positions, while we choose the allowed depth by checking predictions on hidden farms.

## 9.2 Try one validation round first

We have **81 training farms** and **21 final test farms**. To choose depth, split only the 81 training farms into five batches, called **folds**. All sampling rounds from a farm stay together.

**Supplied assignment — `training-folds`:** run this block to assign those batches reproducibly. The displayed counts describe each batch when it is hidden:

In [4]:
RNGkind(kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
set.seed(2027)
training_farms <- sort(unique(train_survey$FarmID), method = "radix")
fold_labels <- sample(rep(1:5, length.out = length(training_farms)))
fold_table <- data.frame(FarmID = training_farms, fold = fold_labels)
row_fold <- fold_table$fold[match(train_survey$FarmID, fold_table$FarmID)]
stopifnot(!any(fold_table$FarmID %in% tree_input$test$FarmID))
data.frame(fold = 1:5, validation_farms = as.integer(table(fold_labels)),
           validation_records = as.integer(table(row_fold)))

fold,validation_farms,validation_records
<int>,<int>,<int>
1,17,84
2,16,86
3,16,86
4,16,83
5,16,81


Start with batch 1. **Validation** means checking predictions for records that did not fit this particular tree. Run `one-round-data`:

In [5]:
fitting <- train_survey[row_fold != 1, ]
validation <- train_survey[row_fold == 1, ]
data.frame(role = c("Fit", "Validate"), records = c(nrow(fitting), nrow(validation)),
           farms = c(length(unique(fitting$FarmID)), length(unique(validation$FarmID))))

role,records,farms
<chr>,<int>,<int>
Fit,336,64
Validate,84,17


Read the table: **336 records from 64 farms** fit this tree; **84 records from 17 other training farms** check it. These 17 are temporarily hidden training farms, not any of the 21 final test farms.

```mermaid
flowchart LR
    accTitle: One Training Validation Round
    accDescr: In round one, four batches with 64 training farms fit a tree and learn missing-input references. The tree predicts a hidden batch of 17 training farms, whose outcomes are used only to calculate prediction errors.
    fitting_farms["64 farms: learn medians and tree"] --> hidden_inputs["17 hidden farms: supply inputs"]
    hidden_inputs --> predictions["Make predictions"]
    predictions --> compare_answers["Then compare with their MSC answers"]
```

**Missing inputs:** learn the medians from the 336 fitting records, then use those same references in both tables. For example, if known fitting counts were 30, 40 and 200, the reference would be 40 even if a hidden farm reported 1,200. Its value cannot redefine the reference.

Run the supplied `one-round-prepare` block. It repeats Chapter 8's median-and-copy operations and displays the references learned for this round:

In [6]:
# Same median rule as Chapter 8, now learned only from this round's fitting records.
round_medians <- learn_medians(fitting)
fitting_ready <- fill_predictors(fitting, round_medians)
validation_ready <- fill_predictors(validation, round_medians)
data.frame(feature = tree_features, fitting_median = unname(round_medians))

feature,fitting_median
<chr>,<dbl>
pasture_hours,9.0
milking_cows,77.5


The cow reference is now **77.5**, rather than Chapter 8's 72, because this round has fewer fitting farms. It is the average of the two middle counts, a computed fill value rather than a measured half cow.

Now the familiar fitting step. Run `one-round-fit`:

In [7]:
round_settings <- fixed_tree_settings
round_settings$maxdepth <- 1
round_tree <- rpart(log10_msc ~ pasture_hours + milking_cows,
                    data = fitting_ready, method = "anova", control = round_settings)

**Find:** which table appears after `data =`? It must be `fitting_ready`, not the hidden validation table.

Run `one-round-predict` to apply that tree to the hidden inputs:

In [8]:
round_predictions <- predict(round_tree, newdata = validation_ready)
head(data.frame(observed_log10 = validation_ready$log10_msc,
                predicted_log10 = as.numeric(round_predictions)), 3)

,observed_log10,predicted_log10
,<dbl>,<dbl>
1,0.5440680,0.6842116
2,0.1760913,0.6842116
3,0.7403627,0.6842116


The observed values appear only to check the predictions. The fitting call did not use them. In `predict()`, the formula's right-hand inputs determine the prediction; an available answer column is not automatically another predictor.

Finally, run `one-round-error` to summarize this round's errors with Chapter 7's RMSE calculation:

In [9]:
round_errors <- validation_ready$log10_msc - round_predictions
sqrt(mean(round_errors^2))

[1] 0.692969

The RMSE of about **0.693** describes only this hidden batch and this depth. The depth choice below uses all five batches.

**Say the sequence:** learn from four batches → predict the fifth → compare with its answers. No final test score has entered this process.

## 9.3 Rotate the hidden batch

We do not want the choice to depend only on batch 1. Repeat the same steps until each training batch has been hidden once:

| Round | Batch 1 | Batch 2 | Batch 3 | Batch 4 | Batch 5 |
|---|---|---|---|---|---|
| 1 | Hide | Fit | Fit | Fit | Fit |
| 2 | Fit | Hide | Fit | Fit | Fit |
| 3 | Fit | Fit | Hide | Fit | Fit |
| 4 | Fit | Fit | Fit | Hide | Fit |
| 5 | Fit | Fit | Fit | Fit | Hide |

For each depth, every training record now has **one prediction made while its farm was hidden**. We call these **out-of-fold (OOF) predictions**. The 21 final test farms do not appear anywhere in this table.

Run the supplied `validate-depth` block. It automates the one-round procedure for all five batches and all three depths. You do not need to write nested loops to understand what is being repeated.

In [10]:
# Repeat the illustrated fit/predict steps for all five folds and three depths.
tree_formula <- log10_msc ~ pasture_hours + milking_cows
oof <- matrix(NA_real_, nrow(train_survey), length(candidate_depths))
oof_baseline <- rep(NA_real_, nrow(train_survey))
fold_medians <- list()
validation_nodes <- list()
for (fold in 1:5) {
  fitting <- train_survey[row_fold != fold, ]
  validation <- train_survey[row_fold == fold, ]
  stopifnot(length(intersect(fitting$FarmID, validation$FarmID)) == 0)
  medians <- learn_medians(fitting)
  fold_medians[[fold]] <- data.frame(fold = fold, feature = tree_features, median = unname(medians))
  fitting <- fill_predictors(fitting, medians)
  validation <- fill_predictors(validation, medians)
  oof_baseline[row_fold == fold] <- mean(fitting$log10_msc)
  for (j in seq_along(candidate_depths)) {
    candidate <- rpart(tree_formula, data = fitting, method = "anova",
                       control = tree_control(candidate_depths[j]))
    candidate_nodes <- extract_tree_nodes(candidate)
    candidate_nodes$fold <- fold
    candidate_nodes$depth <- candidate_depths[j]
    validation_nodes[[length(validation_nodes) + 1L]] <- candidate_nodes
    oof[row_fold == fold, j] <- predict(candidate, newdata = validation)
  }
}
stopifnot(all(is.finite(oof)), all(is.finite(oof_baseline)))
observed <- train_survey$log10_msc
validation_scores <- data.frame(depth = candidate_depths,
  OOF_RMSE_log10 = vapply(seq_along(candidate_depths), function(j) sqrt(mean((observed - oof[, j])^2)), numeric(1)),
  OOF_MAE_log10 = vapply(seq_along(candidate_depths), function(j) mean(abs(observed - oof[, j])), numeric(1)))
baseline_validation <- data.frame(model = "Training-fold mean", depth = 0,
  OOF_RMSE_log10 = sqrt(mean((observed - oof_baseline)^2)), OOF_MAE_log10 = mean(abs(observed - oof_baseline)))

The final OOF RMSE is calculated from all 420 hidden-record errors together, giving each record equal weight. Batch sizes differ, so it is not an unweighted average of the five RMSEs.

## 9.4 Let the validation table choose depth

Before this comparison, we fixed the rule: choose the **smallest OOF RMSE**, with a shallower tree winning a tie. MAE is also shown, but does not select depth.

Run the supplied `choose-depth` block:

In [11]:
chosen_depth <- min(validation_scores$depth[validation_scores$OOF_RMSE_log10 <= min(validation_scores$OOF_RMSE_log10) + 1e-12])
validation_scores$selected <- validation_scores$depth == chosen_depth
validation_scores
baseline_validation
chosen_depth

depth,OOF_RMSE_log10,OOF_MAE_log10,selected
<int>,<dbl>,<dbl>,<lgl>
1,0.7219814,0.5646934,TRUE
2,0.7268655,0.5645752,FALSE
3,0.7423389,0.5842456,FALSE


model,depth,OOF_RMSE_log10,OOF_MAE_log10
<chr>,<dbl>,<dbl>,<dbl>
Training-fold mean,0,0.7206158,0.5574663


[1] 1

**Read it yourself:** depth 1 has OOF RMSE about 0.722, depth 2 about 0.727 and depth 3 about 0.742. Which is smallest? `selected = TRUE` marks that row. A slightly smaller MAE for depth 2 does not change the predeclared RMSE rule.

The baseline row below uses the fitting mean from each round to predict that round's hidden records. Its OOF RMSE is about 0.721. Choosing the best of three trees does not establish that it beats this simpler reference.

Do not compare 0.722 here directly with Chapter 7's test RMSE of 0.529: they concern different farms. Compare methods on the **same hidden records**.

## 9.5 Revisit your guess, then fit on all training farms

Run the supplied `training-check` block. It calculates the full-training medians and input copy again, then compares fit-to-training errors with OOF errors:

In [12]:
# An illustration after selection: these fits do not alter the chosen depth.
training_medians <- learn_medians(train_survey)
train_filled <- fill_predictors(train_survey, training_medians)
validation_scores$training_RMSE_log10 <- vapply(candidate_depths, function(depth) {
  fitted <- rpart(tree_formula, data = train_filled, method = "anova", control = tree_control(depth))
  sqrt(mean((train_filled$log10_msc - predict(fitted, newdata = train_filled))^2))
}, numeric(1))
validation_scores[c("depth", "training_RMSE_log10", "OOF_RMSE_log10", "selected")]

depth,training_RMSE_log10,OOF_RMSE_log10,selected
<int>,<dbl>,<dbl>,<lgl>
1,0.7014369,0.7219814,TRUE
2,0.6878793,0.7268655,FALSE
3,0.6767658,0.7423389,FALSE


**Return to your guess:** as allowed depth increases, training RMSE falls from about 0.701 to 0.677, but hidden-farm RMSE rises from about 0.722 to 0.742. More detailed groups fitted the known answers better without improving the hidden predictions here.

The extra training fits illustrate this difference; they do not alter the OOF choice. Depth is already selected. Now use **all 420 records from the 81 training farms** to learn the final tree. The medians again come from those training records only. Run `fit-tree`:

In [13]:
selected_settings <- fixed_tree_settings
selected_settings$maxdepth <- chosen_depth
final_tree <- rpart(log10_msc ~ pasture_hours + milking_cows,
                    data = train_filled, method = "anova", control = selected_settings)

There is no single validation-round tree to keep: each round had fewer fitting farms. This final fit uses all available training farms and the chosen setting. `final_tree` stores its learned branches and leaf means. We will use it on the 21 test farms in Chapter 10.

## 9.6 Save the choice

Run the supplied `save-tree-fit` block:

In [14]:
write.csv(fold_table, "outputs/tree/training-farm-folds.csv", row.names = FALSE)
write.csv(do.call(rbind, fold_medians), "outputs/tree/fold-medians.csv", row.names = FALSE)
write.csv(do.call(rbind, validation_nodes), "outputs/tree/validation-nodes.csv", row.names = FALSE)
write.csv(validation_scores, "outputs/tree/validation-scores.csv", row.names = FALSE)
write.csv(baseline_validation, "outputs/tree/baseline-validation.csv", row.names = FALSE)
oof_table <- data.frame(FarmID = train_survey$FarmID, Sampling = train_survey$Sampling,
  fold = row_fold, observed_log10 = observed, baseline_prediction = oof_baseline,
  depth_1 = oof[, 1], depth_2 = oof[, 2], depth_3 = oof[, 3])
write.csv(oof_table, "outputs/tree/out-of-fold-predictions.csv", row.names = FALSE)
saveRDS(list(fit = final_tree, medians = training_medians, chosen_depth = chosen_depth,
             train = train_survey, test = tree_input$test, audit = tree_input$audit,
             validation = validation_scores, baseline_validation = baseline_validation),
        "outputs/checkpoints/tree-fit.rds")
cat("Saved tree-fit.rds; no tree test score has been calculated.\n")

Saved tree-fit.rds; no tree test score has been calculated.


**Find:** `outputs/checkpoints/tree-fit.rds`, `outputs/tree/validation-scores.csv` and `outputs/tree/training-farm-folds.csv`.

**Explain before moving on:** why are validation farms still part of the training set overall? Why do we fit once more after choosing depth? Which farms must stay out of both choices?

```{admonition} Check your explanation
:class: tip dropdown
Validation farms belong to the original 81 training farms: each batch is temporarily hidden while the others fit a candidate. After choosing depth, fit again on all 81 training farms because each round used fewer farms. The 21 final test farms help neither choose depth nor learn the tree or missing-input references.
```

**Next:** [read the tree, predict and compare](10-test-tree.ipynb).